In [1]:
import json

import re

import aiida
from aiida import orm

from tqdm import tqdm

from mc3d_web_pipeline.utils import get_mc3d_id

AIIDA_PROFILE_NAME = "mc3d-pbesol-v1-fermisurf"

aiida.load_profile(AIIDA_PROFILE_NAME)

Profile<uuid='232a7a8f9402451994f91e73b35a018e' name='mc3d-pbesol-v1-fermisurf'>

In [2]:
# list aiida groups
! verdi -p {AIIDA_PROFILE_NAME} group list --all --all-users --count

  PK  Label            Type string    User               Node count
----  ---------------  -------------  ---------------  ------------
   5  20251215-114125  core.import    aiida@localhost         67289
   4  bxsf             core           aiida@localhost           407
   1  skeaf_001_100    core           aiida@localhost           407
   3  skeaf_100_010    core           aiida@localhost           407
   2  skeaf_110_001    core           aiida@localhost           407


In [3]:
g = orm.load_group("bxsf")
n = g.nodes[0]
n.uuid

'00bb1857-013b-4847-ab09-3e4bc506a3c5'

In [8]:
! verdi node graph generate 00bb1857-013b-4847-ab09-3e4bc506a3c5

/home/eimre_k/micromamba/envs/mc3d-web-pipeline/lib/python3.10/site-packages/aiida/orm/nodes/data/code/legacy.py:585: AiidaDeprecationWarning: `Code.get_execname` method is deprecated, use `get_executable` instead. (this will be removed in v3)
  warn_deprecation('`Code.get_execname` method is deprecated, use `get_executable` instead.', version=3)
/home/eimre_k/micromamba/envs/mc3d-web-pipeline/lib/python3.10/site-packages/aiida/orm/nodes/data/code/legacy.py:556: AiidaDeprecationWarning: `Code.is_local` method is deprecated, use a `PortableCode` instance and check the type. (this will be removed in v3)
  warn_deprecation(
/home/eimre_k/micromamba/envs/mc3d-web-pipeline/lib/python3.10/site-packages/aiida/orm/nodes/data/code/legacy.py:507: AiidaDeprecationWarning: `Code.get_remote_exec_path` method is deprecated, use `InstalledCode.filepath_executable` instead. (this will be removed in v3)
  warn_deprecation(
Success: Output written to `4003.dot.pdf`


In [4]:
qb = (
    orm.QueryBuilder()
    # BSXF CalcJobs from the group
    .append(orm.Group, filters={"label": "bxsf"}, tag="group")
    .append(orm.CalcJobNode, with_group="group", project=["*"], tag="bxsf_cj")
    # Go up to get the parent W90 CalcJob
    .append(orm.RemoteData, with_outgoing="bxsf_cj", tag="u1")
    .append(orm.CalcFunctionNode, with_outgoing="u1", tag="u2")
    .append(orm.RemoteStashFolderData, with_outgoing="u2", tag="u3")
    .append(orm.CalcJobNode, with_outgoing="u3", project=["*"], tag="w90_cj")
    # Go down the BSXF outputs to reach the Skeaf workchains
    .append(orm.RemoteStashFolderData, with_incoming="bxsf_cj", tag="d1")
    .append(orm.CalcFunctionNode, with_incoming="d1", tag="d2")
    .append(orm.RemoteData, with_incoming="d2", tag="d3")
    .append(orm.WorkChainNode, with_incoming="d3", project=["*"], tag="skeaf_wc")
)

count = qb.distinct().count()
print(f"Number: {count}")

Number: 1221


In [5]:
collected_data = {}

for bxsf_cj, w90_cj, skeaf_wc in tqdm(qb.iterall(), total=count):   
    if bxsf_cj.uuid not in collected_data:
        collected_data[bxsf_cj.uuid] = {
            'bxsf_cj_node': bxsf_cj,
            'w90_cj_node': w90_cj,
            'skeaf_wc_nodes': [],
        }
    collected_data[bxsf_cj.uuid]['skeaf_wc_nodes'].append(skeaf_wc)

  0%|          | 0/1221 [00:00<?, ?it/s]

100%|██████████| 1221/1221 [00:00<00:00, 2881.11it/s]


In [6]:
def get_skeaf_bands_formatted(skeaf_outputs):
    """
    A skeaf workchain creates an output node for each band that crosses the fermi level.
    Collect all of these into a dictionary.
    """
    if "skeaf" not in skeaf_outputs:
        print("Warning: No skeaf outputs!")
        return []
    bands_formatted = []
    for band_label in skeaf_outputs.skeaf:
        bands_formatted.append({
            "band_number": int(re.search(r"\d+", band_label).group()),
            "frequency_arraydata_uuid": skeaf_outputs.skeaf[band_label]['frequency'].uuid,
            "output_parameters_uuid": skeaf_outputs.skeaf[band_label]['output_parameters'].uuid,
            "fermi_energy": skeaf_outputs.skeaf[band_label]['output_parameters']['fermi_energy'],
        })
    return sorted(bands_formatted, key=lambda x: x['band_number'])

def get_skeaf_formatted(skeaf_wc):
    return {
        "skeaf_wc_uuid": skeaf_wc.uuid,
        "mf_path": skeaf_wc.base.extras.all['mf_path'],
        "input_params_uuid": skeaf_wc.inputs.skeaf.parameters.uuid,
        "bands": get_skeaf_bands_formatted(skeaf_wc.outputs)
    }

def get_formatted_entry(entry_data):

    bxsf_cj_node = entry_data['bxsf_cj_node']
    w90_cj_node = entry_data['w90_cj_node']
    skeaf_wc_nodes = entry_data['skeaf_wc_nodes']
    print(entry_data)

    return {
        'wannier90_calcjob_uuid': w90_cj_node.uuid,
        'wannierjl_calcjob_uuid': bxsf_cj_node.uuid,
        'skeaf_workchains': [get_skeaf_formatted(skeaf_wc) for skeaf_wc in skeaf_wc_nodes],
    }


In [7]:
formatted_entries = {}

for bxsf_cj_uuid, entry_data in collected_data.items():

    bxsf_cj_node = entry_data['bxsf_cj_node']
    mc3d_id = get_mc3d_id({
        "id": bxsf_cj_node.base.extras.all['source_id'],
        "database": bxsf_cj_node.base.extras.all['source_db']
    })

    assert mc3d_id is not None

    formatted_entries[mc3d_id] = get_formatted_entry(entry_data)

with open("../base.json", "w") as fh:
    json.dump(formatted_entries, fh, indent=2)

{'bxsf_cj_node': <CalcJobNode: uuid: 00bb1857-013b-4847-ab09-3e4bc506a3c5 (pk: 4003) (aiida.calculations:fermisurface.wannierjl)>, 'w90_cj_node': <CalcJobNode: uuid: 5ada7ddf-5019-47b4-83b8-2a129546a00e (pk: 1374) (aiida.calculations:wannier90.wannier90)>, 'skeaf_wc_nodes': [<WorkChainNode: uuid: afd282e8-df77-4d3a-80f5-40bd05d9b06a (pk: 48737) (aiida.workflows:skeaf.skeaf)>, <WorkChainNode: uuid: db8eb013-7710-45e5-8c8e-3d22a33b0965 (pk: 11852) (aiida.workflows:skeaf.skeaf)>, <WorkChainNode: uuid: 0ff6f1a0-2974-4b9d-9068-90ba694105db (pk: 29115) (aiida.workflows:skeaf.skeaf)>]}


{'bxsf_cj_node': <CalcJobNode: uuid: 02e2fd6b-5f5b-4303-804e-0522b8583004 (pk: 4018) (aiida.calculations:fermisurface.wannierjl)>, 'w90_cj_node': <CalcJobNode: uuid: e6e8ede7-1414-48b4-9a08-8868992a01b7 (pk: 1944) (aiida.calculations:wannier90.wannier90)>, 'skeaf_wc_nodes': [<WorkChainNode: uuid: fe30ca1e-5029-4814-aa69-66d6fbcc230d (pk: 49940) (aiida.workflows:skeaf.skeaf)>, <WorkChainNode: uuid: 0740cd00-3e70-4a13-951c-5845c9042cef (pk: 10054) (aiida.workflows:skeaf.skeaf)>, <WorkChainNode: uuid: 77cf9214-f02e-4660-b700-970a3adfb6c7 (pk: 29584) (aiida.workflows:skeaf.skeaf)>]}
{'bxsf_cj_node': <CalcJobNode: uuid: 04478683-0756-40b8-8a00-b5826a611ba3 (pk: 4022) (aiida.calculations:fermisurface.wannierjl)>, 'w90_cj_node': <CalcJobNode: uuid: 2b310ff7-3f4e-4ab9-b493-f3371bb8fa0c (pk: 243) (aiida.calculations:wannier90.wannier90)>, 'skeaf_wc_nodes': [<WorkChainNode: uuid: 7181b69d-2656-4ed1-ab94-5e906e1f97d6 (pk: 9323) (aiida.workflows:skeaf.skeaf)>, <WorkChainNode: uuid: 4e1bfb64-1b68-4

In [39]:
n = orm.load_node("509c20d3-6c58-40a0-82f3-8759e79495d9")

In [41]:
list(n.outputs)

['wan2skeaf']

In [36]:
n.outputs.skeaf['band10']

AttributeDict({'retrieved': <FolderData: uuid: 6863fa60-7843-4f05-a22b-9fcfe8954a9c (pk: 18430)>, 'remote_folder': <RemoteData: uuid: 93453656-c69c-4e3c-a16d-214121ba27b3 (pk: 9475)>, 'output_parameters': <Dict: uuid: 27f23de9-75ff-4578-98cb-69fca9122808 (pk: 17142)>, 'frequency': <ArrayData: uuid: acbeb38e-2ac2-4b3a-b12f-e5265d61b436 (pk: 18692)>})